In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, current_timestamp



In [ ]:
# ==============================================================================
# PHASE 2: Enterprise Data Pipeline (Mock Central Catalog -> Databricks -> Unity Catalog)


In [ ]:
# ==============================================================================
print("Initializing Phase 2 Pipeline...")

# Initialize a unified Spark session with both Iceberg and Delta Lake capabilities.
# This simulates the rich Databricks runtime environment.
spark = SparkSession.builder \
    .appName("Databricks-POC-Simulator") \
    .config("spark.jars.packages", 
            "io.unitycatalog:unitycatalog-spark_2.12:0.2.0,"
            "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0,"
            "org.apache.iceberg:iceberg-aws-bundle:1.5.0,"
            "io.delta:delta-spark_2.12:3.2.1,"
            "io.delta:delta-iceberg_2.12:3.2.1," \
            "org.apache.hadoop:hadoop-aws:3.3.4") \
    .config("spark.sql.extensions", 
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,"
            "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.sql.catalog.iceberg_cat", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.iceberg_cat.type", "hive") \
    .config("spark.sql.catalog.unity", "io.unitycatalog.spark.UCSingleCatalog") \
    .config("spark.sql.catalog.unity.warehouse", "s3a://lakehouse-bucket/unity_catalog/") \
    .config("spark.sql.catalog.unity.uri", "http://unity-catalog-server:8080") \
    .config("spark.sql.catalog.central_catalog", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.central_catalog.catalog-impl", "org.apache.iceberg.rest.RESTCatalog") \
    .config("spark.sql.catalog.central_catalog.uri", "http://iceberg-rest:8181") \
    .config("spark.sql.catalog.central_catalog.io-impl", "org.apache.iceberg.aws.s3.S3FileIO") \
    .config("spark.sql.catalog.central_catalog.warehouse", "s3a://lakehouse-bucket/central_warehouse/") \
    .config("spark.hadoop.hive.metastore.schema.verification", "false") \
    .config("spark.hadoop.hive.support.concurrency", "false") \
    .config("spark.hadoop.hive.txn.manager", "org.apache.hadoop.hive.ql.lockmgr.DummyTxnManager") \
    .config("spark.hadoop.datanucleus.schema.autoCreateAll", "true") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://localstack:4566") \
    .config("spark.sql.catalog.central_catalog.s3.endpoint", "http://localstack:4566") \
    .config("spark.sql.catalog.central_catalog.s3.path-style-access", "true") \
    .config("spark.sql.catalog.central_catalog.client.region", "us-east-1") \
    .config("spark.sql.catalog.central_catalog.s3.access-key-id", "test") \
    .config("spark.sql.catalog.central_catalog.s3.secret-access-key", "test") \
    .config("spark.hadoop.fs.s3a.access.key", "test") \
    .config("spark.hadoop.fs.s3a.secret.key", "test") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .getOrCreate()



In [ ]:
# ------------------------------------------------------------------------------
# Phase 2A: Originate Central Data (Iceberg REST Mock)


In [ ]:
# ------------------------------------------------------------------------------
print("\n--- [Phase 2A] Originating Source Data in Central Catalog ---")
spark.sql("CREATE NAMESPACE IF NOT EXISTS central_catalog.default")

data = [("1", "Alice", 1000), ("2", "Bob", 2000), ("3", "Charlie", 3000)]
columns = ["account_id", "name", "balance"]
df_source = spark.createDataFrame(data, columns)

df_source.writeTo("central_catalog.default.source_accounts").using("iceberg").createOrReplace()
print("Successfully created 'source_accounts' in Central Catalog (Iceberg).")



In [ ]:
# ------------------------------------------------------------------------------
# Phase 2B & 2C: Federated Read & Data Manipulation


In [ ]:
# ------------------------------------------------------------------------------
print("\n--- [Phase 2B & 2C] Federated Read & Data Manipulation (Databricks) ---")
# Simulating Databricks reading the Central table via federation
df_federated = spark.table("central_catalog.default.source_accounts")

df_transformed = df_federated \
    .withColumn("balance_with_interest", col("balance") * 1.05) \
    .withColumn("processed_at", current_timestamp())

print("Data transformation complete. Schema:")
df_transformed.printSchema()



In [ ]:
# ------------------------------------------------------------------------------
# Phase 2D: Write to Unity Catalog (Delta UniForm)


In [ ]:
# WORKAROUND: Create missing Hive transaction tables in Derby so Iceberg's stubborn HiveCatalog succeeds
def create_derby_tables(spark):
    try:
        sc = spark.sparkContext
        DriverManager = sc._gateway.jvm.java.sql.DriverManager
        conn = DriverManager.getConnection("jdbc:derby:/home/jovyan/work/metastore_db;create=true")
        stmt = conn.createStatement()
        queries = [
            "CREATE TABLE NEXT_LOCK_ID (NL_NEXT BIGINT NOT NULL)",
            "INSERT INTO NEXT_LOCK_ID VALUES (1)",
            "CREATE TABLE HIVE_LOCKS (HL_LOCK_EXT_ID BIGINT NOT NULL, HL_LOCK_INT_ID BIGINT NOT NULL, HL_TXNID BIGINT NOT NULL, HL_DB VARCHAR(128) NOT NULL, HL_TABLE VARCHAR(128), HL_PARTITION VARCHAR(767), HL_LOCK_STATE CHAR(1) NOT NULL, HL_LOCK_TYPE CHAR(1) NOT NULL, HL_LAST_HEARTBEAT BIGINT NOT NULL, HL_ACQUIRED_AT BIGINT, HL_USER VARCHAR(128) NOT NULL, HL_HOST VARCHAR(128) NOT NULL, HL_HEARTBEAT_COUNT INT, HL_AGENT_INFO VARCHAR(128), HL_BLOCKEDBY_EXT_ID BIGINT, HL_BLOCKEDBY_INT_ID BIGINT, PRIMARY KEY(HL_LOCK_EXT_ID, HL_LOCK_INT_ID))",
            "CREATE TABLE TXNS (TXN_ID BIGINT NOT NULL, TXN_STATE CHAR(1) NOT NULL, TXN_STARTED BIGINT NOT NULL, TXN_LAST_HEARTBEAT BIGINT NOT NULL, TXN_USER VARCHAR(128) NOT NULL, TXN_HOST VARCHAR(128) NOT NULL, TXN_AGENT_INFO VARCHAR(128), TXN_META_INFO VARCHAR(128), TXN_HEARTBEAT_COUNT INT, TXN_TYPE INT, PRIMARY KEY(TXN_ID))",
            "CREATE TABLE NEXT_TXN_ID (NTXN_NEXT BIGINT NOT NULL)",
            "INSERT INTO NEXT_TXN_ID VALUES (1)",
            "CREATE TABLE AUX_TABLE (MT_KEY1 VARCHAR(128) NOT NULL, MT_KEY2 BIGINT NOT NULL, MT_COMMENT VARCHAR(255), PRIMARY KEY(MT_KEY1, MT_KEY2))",
            "CREATE TABLE WRITE_SET (WS_DATABASE VARCHAR(128) NOT NULL, WS_TABLE VARCHAR(128) NOT NULL, WS_PARTITION VARCHAR(767), WS_TXNID BIGINT NOT NULL, WS_COMMIT_ID BIGINT NOT NULL, WS_OPERATION_TYPE CHAR(1) NOT NULL)",
            "CREATE TABLE TXN_COMPONENTS (TC_TXNID BIGINT NOT NULL, TC_DATABASE VARCHAR(128) NOT NULL, TC_TABLE VARCHAR(128), TC_PARTITION VARCHAR(767), TC_OPERATION_TYPE CHAR(1) NOT NULL, TC_WRITEID BIGINT)",
            "CREATE TABLE COMPLETED_TXN_COMPONENTS (CTC_TXNID BIGINT NOT NULL, CTC_DATABASE VARCHAR(128) NOT NULL, CTC_TABLE VARCHAR(256), CTC_PARTITION VARCHAR(767), CTC_TIMESTAMP timestamp DEFAULT CURRENT_TIMESTAMP NOT NULL, CTC_WRITEID BIGINT, CTC_UPDATE_DELETE CHAR(1) NOT NULL)",
            "CREATE TABLE COMPACTION_QUEUE (CQ_ID BIGINT NOT NULL, CQ_DATABASE VARCHAR(128) NOT NULL, CQ_TABLE VARCHAR(128) NOT NULL, CQ_PARTITION VARCHAR(767), CQ_STATE CHAR(1) NOT NULL, CQ_TYPE CHAR(1) NOT NULL, CQ_TBLPROPERTIES VARCHAR(2048), CQ_WORKER_ID VARCHAR(128), CQ_START BIGINT, CQ_RUN_AS VARCHAR(128), CQ_HIGHEST_WRITE_ID BIGINT, CQ_META_INFO VARCHAR(2048) FOR BIT DATA, CQ_HADOOP_JOB_ID VARCHAR(32), PRIMARY KEY(CQ_ID))",
            "CREATE TABLE COMPLETED_COMPACTIONS (CC_ID BIGINT NOT NULL, CC_DATABASE VARCHAR(128) NOT NULL, CC_TABLE VARCHAR(128) NOT NULL, CC_PARTITION VARCHAR(767), CC_STATE CHAR(1) NOT NULL, CC_TYPE CHAR(1) NOT NULL, CC_TBLPROPERTIES VARCHAR(2048), CC_WORKER_ID VARCHAR(128), CC_START BIGINT, CC_END BIGINT, CC_RUN_AS VARCHAR(128), CC_HIGHEST_WRITE_ID BIGINT, CC_META_INFO VARCHAR(2048) FOR BIT DATA, CC_HADOOP_JOB_ID VARCHAR(32), PRIMARY KEY(CC_ID))",
            "CREATE TABLE NEXT_COMPACTION_QUEUE_ID (NCQ_NEXT BIGINT NOT NULL)",
            "INSERT INTO NEXT_COMPACTION_QUEUE_ID VALUES (1)",
            "CREATE TABLE TXN_TO_WRITE_ID (T2W_TXNID BIGINT NOT NULL, T2W_DATABASE VARCHAR(128) NOT NULL, T2W_TABLE VARCHAR(256) NOT NULL, T2W_WRITEID BIGINT NOT NULL)",
            "CREATE TABLE NEXT_WRITE_ID (NWI_DATABASE VARCHAR(128) NOT NULL, NWI_TABLE VARCHAR(256) NOT NULL, NWI_NEXT BIGINT NOT NULL)",
            "CREATE TABLE MIN_HISTORY_LEVEL (MHL_TXNID BIGINT NOT NULL, MHL_MIN_OPEN_TXNID BIGINT NOT NULL, PRIMARY KEY(MHL_TXNID))",
            "CREATE TABLE MATERIALIZATION_REBUILD_LOCKS (MRL_TXN_ID BIGINT NOT NULL, MRL_DB_NAME VARCHAR(128) NOT NULL, MRL_TBL_NAME VARCHAR(256) NOT NULL, MRL_LAST_HEARTBEAT BIGINT NOT NULL, PRIMARY KEY(MRL_TXN_ID))",
            "CREATE TABLE IUD_CACHE (IC_DB VARCHAR(128) NOT NULL, IC_TABLE VARCHAR(128) NOT NULL, IC_PARTITION VARCHAR(767), IC_DELETED_RECORDS VARCHAR(4000), IC_UPDATED_RECORDS VARCHAR(4000))"
        ]
        for q in queries:
            try:
                stmt.executeUpdate(q)
            except:
                pass
        try:
            stmt.executeUpdate("DELETE FROM HIVE_LOCKS")
            stmt.executeUpdate("DELETE FROM TXNS")
        except:
            pass
        print("Created missing Derby tables for Hive lock manager.")
    except Exception as e:
        print("Failed to connect to Derby:", e)

create_derby_tables(spark)


In [ ]:
# ------------------------------------------------------------------------------
print("\n--- [Phase 2D] Writing to Unity Catalog ---")
uc_table_path = "s3://lakehouse-bucket/unity_catalog/transformed_accounts"

# 1. Write plain Delta to S3 (no UniForm, no column mapping UUIDs!)
df_transformed.write.format("delta").mode("overwrite").save(uc_table_path)
print("Successfully wrote raw Delta files to S3.")

# 2. Register directly in Unity Catalog via REST API
print("\nRegistering table in Unity Catalog via REST API...")
import requests
schema = df_transformed.schema

def map_type_name(t):
    name = t.typeName().upper()
    if name == 'INTEGER':
        return 'INT'
    return name

columns = [
    {
        "name": f.name,
        "type_text": f.dataType.simpleString(),
        "type_json": f.json(),
        "type_name": map_type_name(f.dataType),
        "position": i,
        "nullable": f.nullable,
    }
    for i, f in enumerate(schema.fields)
]

resp = requests.post(
    "http://unity-catalog-server:8080/api/2.1/unity-catalog/tables",
    json={
        "name": "transformed_accounts",
        "catalog_name": "unity",
        "schema_name": "default",
        "table_type": "EXTERNAL",
        "data_source_format": "DELTA",
        "storage_location": uc_table_path,
        "columns": columns,
    },
)
print(f"Unity Catalog Registration Status: {resp.status_code}")
if resp.status_code not in (200, 201):
    print(f"Response: {resp.text}")
else:
    print(f"Successfully registered standard Delta table in Unity Catalog at {uc_table_path}.")

spark.stop()
print("\nPhase 2 Complete.")
